# PrediCT COCA — End-to-End Wrapper Notebook (Colab)

This single notebook runs the pipeline **step-by-step** so you can debug issues (missing DICOM files, empty patient folders, Google Drive placeholders) before training.

## What you will run

1. **Setup**: install dependencies, mount Drive, set `PREDICT_RAW_DIR`
2. **Generate metadata**: scan patient folders → `data/metadata_all.csv`
3. **Validate DICOM**: build a cleaned list → `data/metadata_clean*.csv` and a report CSV
4. **Export processed volumes**: read DICOM → resample → HU window → write `.npy`
5. **DataLoader sanity check**: load a batch
6. **Training stub**: verify GPU training works

## Evaluation deliverables (Common Task)

This notebook generates the required artifacts:

- Dataset statistics: `outputs/dataset_stats.json`
- Train/val/test split manifest: `outputs/splits.json`
- Processed volume manifest: `outputs/processed_manifest.csv`
- Short written justification: `outputs/justification.txt`

## What to expect

- DICOM reading uses **SimpleITK on CPU**.
- Exporting `.npy` volumes is mostly **CPU-bound** (I/O + resampling).
- Training uses **GPU (CUDA)** if available.

## Colab GPU

Runtime → Change runtime type → Hardware accelerator: **GPU**.

## Step 1 — Setup

Upload the repository folder to Colab

Upload the folder in the left Files pane, then `cd` into it.

In [ ]:
import os
os.chdir('/content')

In [ ]:
from google.colab import userdata
import os
import subprocess

TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f"https://{TOKEN}@github.com/vrAxiom/PrediCT-CLI.git"
repo_dir = 'PrediCT'

# Ensure we are in the root /content directory
os.chdir('/content')

if os.path.exists(repo_dir):
    print(f"Directory '{repo_dir}' already exists. Performing git pull.")
    os.chdir(repo_dir)
    subprocess.run(['git', 'pull'], check=True)
else:
    print(f"Cloning repository '{repo_dir}'.")
    subprocess.run(['git', 'clone', '-q', REPO_URL, repo_dir], check=True)
    os.chdir(repo_dir)

subprocess.run(['pip', '-q', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['pip', '-q', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run(['pip', '-q', 'install', '-e', '.'], check=True)
import sys
sys.path.insert(0, os.path.join('/content', repo_dir, 'src'))
from predict.validate import validate_metadata_csv

In [ ]:
import subprocess
subprocess.run(['pip', '-q', 'install', 'numpy>=1.24', 'scipy>=1.10', 'SimpleITK>=2.3', 'tqdm>=4.65', 'pandas>=2.0', 'torch>=2.5'], check=True)

### Mount Drive and set the COCA raw patient root

Your Windows path is:

`I:\\My Drive\\GSoC_PrediCT\\data_raw\\dicom\\Gated_release_final\\patient`

In Colab, after mounting Drive, the equivalent is typically:

`/content/drive/MyDrive/GSoC_PrediCT/data_raw/dicom/Gated_release_final/patient`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from pathlib import Path

raw_dir = Path('/content/drive/MyDrive/GSoC_PrediCT/data_raw/dicom/Gated_release_final/patient')
os.environ['PREDICT_RAW_DIR'] = str(raw_dir)
print('PREDICT_RAW_DIR=', os.environ['PREDICT_RAW_DIR'])
print('raw_dir_exists=', raw_dir.exists())

PREDICT_RAW_DIR= /content/drive/MyDrive/GSoC_PrediCT/data_raw/dicom/Gated_release_final/patient
raw_dir_exists= True


### GPU check (training uses GPU if available)

Install PyTorch. If `torch.cuda.is_available()` is False, training will run on CPU.

In [ ]:
import torch

print('torch=', torch.__version__)
print('cuda_available=', torch.cuda.is_available())
print('cuda_device_count=', torch.cuda.device_count())
if torch.cuda.is_available():
    print('cuda_name=', torch.cuda.get_device_name(0))

torch= 2.10.0+cpu
cuda_available= False
cuda_device_count= 0


## Step 2 — Generate metadata for all patients

This scans the raw patient folder and creates `data/metadata_all.csv` with one row per patient folder that contains `.dcm` files.

Expected output: ~787 rows.

In [ ]:
import subprocess
from pathlib import Path

out_csv = Path('/content/drive/MyDrive/GSoC_PrediCT/data/metadata_all.csv')
out_csv.parent.mkdir(parents=True, exist_ok=True)

subprocess.check_call(
    [
        'predict',
        'make-metadata',
        '--project-root',
        str(Path.cwd()),
        '--raw-dir',
        str(raw_dir),
        '--out-csv',
        str(out_csv),
        '--default-label',
        '0',
        '--kind',
        'dicom_series',
    ]
)

print(out_csv)
print('bytes=', out_csv.stat().st_size)

/content/drive/MyDrive/GSoC_PrediCT/data/metadata_all.csv
bytes= 18699


In [ ]:
import csv

with out_csv.open('r', newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

print('rows=', len(rows))
rows[:5]

rows= 787


[{'subject_id': '0', 'image': '0', 'label': '0', 'kind': 'dicom_series'},
 {'subject_id': '1', 'image': '1', 'label': '0', 'kind': 'dicom_series'},
 {'subject_id': '10', 'image': '10', 'label': '0', 'kind': 'dicom_series'},
 {'subject_id': '100', 'image': '100', 'label': '0', 'kind': 'dicom_series'},
 {'subject_id': '101', 'image': '101', 'label': '0', 'kind': 'dicom_series'}]

## Step 3 — Validate DICOM series (clean the metadata)

This produces:

- `data/metadata_clean.csv` (keeps only patients that pass validation)
- `outputs/dicom_validation_report.csv` (why patients were excluded)

Validation modes:

- `fast`: only checks that `.dcm` files exist in the folder
- `shallow`: SimpleITK enumerates series files
- `deep`: SimpleITK reads each series (slowest, best for Drive placeholders/corrupt series)


In [ ]:
import subprocess
from pathlib import Path

clean_csv = Path('/content/drive/MyDrive/GSoC_PrediCT/data/metadata_clean.csv')
report_csv = Path('/content/drive/MyDrive/GSoC_PrediCT/outputs/dicom_validation_report.csv')
report_csv.parent.mkdir(parents=True, exist_ok=True)

subprocess.check_call(
    [
        'predict',
        'validate-metadata',
        '--project-root',
        str(Path.cwd()),
        '--raw-dir',
        str(raw_dir),
        '--metadata-csv',
        str(out_csv),
        '--out-clean-csv',
        str(clean_csv),
        '--out-report-csv',
        str(report_csv),
        '--mode',
        'shallow',
    ]
)

clean_csv, report_csv

validate:deep: 100%|██████████| 787/787 [00:00<00:00, 176321.63subject/s]


(PosixPath('/content/drive/MyDrive/GSoC_PrediCT/data/metadata_clean.csv'),
 PosixPath('/content/drive/MyDrive/GSoC_PrediCT/outputs/dicom_validation_report.csv'))

In [ ]:
print("report size =", report_csv.stat().st_size)
print("clean size  =", clean_csv.stat().st_size)

report size = 87968
clean size  = 17911


In [ ]:
import pandas as pd

df = pd.read_csv(report_csv)
print('ok_counts=')
print(df['ok'].value_counts())
df[df['ok'] == False].head(15)
print(df.head())

ok_counts=
ok
True    787
Name: count, dtype: int64
   subject_id                                              image  \
0           0  /content/drive/MyDrive/GSoC_PrediCT/data_raw/d...   
1           1  /content/drive/MyDrive/GSoC_PrediCT/data_raw/d...   
2          10  /content/drive/MyDrive/GSoC_PrediCT/data_raw/d...   
3         100  /content/drive/MyDrive/GSoC_PrediCT/data_raw/d...   
4         101  /content/drive/MyDrive/GSoC_PrediCT/data_raw/d...   

           kind    ok  num_files reason  
0  dicom_series  True         57     ok  
1  dicom_series  True         57     ok  
2  dicom_series  True         46     ok  
3  dicom_series  True         46     ok  
4  dicom_series  True         49     ok  


## Step 4 — Export processed `.npy` volumes

This is where preprocessing actually happens. Note: For performance you can run with PROJECT_ROOT as colab runtime disk but then you need to copy data into your colab directory with following subsequent copy steps.

Outputs:
- `data/processed/<subject_id>.npy`
- `outputs/processed_manifest.csv`
- `outputs/dataset_stats.json`
- `outputs/splits.json`

If you still get errors, use the validation report to identify bad patients and rerun.


In [ ]:
from pathlib import Path

from predict.pipeline import run_pipeline
from predict.config import HUWindowConfig, ResampleConfig

PROJECT_ROOT = Path('/content/drive/MyDrive/GSoC_PrediCT')

resample_cfg = ResampleConfig(mode='spacing', target_spacing=(3.0, 0.7, 0.7))
hu_cfg = HUWindowConfig(lower=-200.0, upper=800.0)

out = run_pipeline(
    project_root=PROJECT_ROOT,
    raw_dir=raw_dir,
    metadata_csv=clean_csv,
    resample_cfg=resample_cfg,
    hu_cfg=hu_cfg,
    export_processed=True,
    dry_run=False,
)

print('processed_manifest_path=', out.processed_manifest_path)
print('warnings_head=', out.stats.get('warnings', [])[:10])

export:test: 100%|██████████| 158/158 [44:50<00:00, 17.03s/subject]


processed_manifest_path= /content/PrediCT/outputs/processed_manifest.csv
warnings_head= []


# Step 4.1
>  For efficiency if you used the local runtime disk to generate outputs . Now is the time to copy to your mounted Google Drive. If you generated out put in Google Drive Project folder directly,  no need to run below copy steps under thsi section Step 4.1.

## Step 3.5 — Visualise Samples for HU Calibration

Before exporting the entire dataset, let's look at a few raw DICOM series. You can use this to hover over pixels (in most notebook environments) or print out intensity statistics for specific regions to decide on your `HUWindowConfig` bounds.

In [ ]:
import SimpleITK as sitk
import matplotlib.pyplot as plt
import numpy as np

def preview_dicom_series(metadata_csv, num_samples=3):
    df_clean = pd.read_csv(metadata_csv)
    samples = df_clean.head(num_samples)

    for i, row in samples.iterrows():
        reader = sitk.ImageSeriesReader()
        dicom_names = reader.GetGDCMSeriesFileNames(row['image'])
        reader.SetFileNames(dicom_names)
        image = reader.Execute()

        # Convert to numpy
        data = sitk.GetArrayFromImage(image)
        mid_slice = data[data.shape[0] // 2]

        plt.figure(figsize=(8, 8))
        plt.imshow(mid_slice, cmap='gray', vmin=-200, vmax=800)
        plt.title(f"Subject: {row['subject_id']} - Mid-slice (HU -200 to 800)")
        plt.colorbar(label='HU Value')
        plt.show()

        print(f"Stats for Subject {row['subject_id']}:")
        print(f"Min HU: {np.min(data)}, Max HU: {np.max(data)}, Mean HU: {np.mean(data):.2f}")
        print("-" * 30)

# Run preview on the cleaned metadata
preview_dicom_series(clean_csv)

In [ ]:
import os
import subprocess

# Ensure we are in the correct directory to list processed files
os.chdir('/content/PrediCT/outputs/')
# copy it there with verbose output to show overwrites
subprocess.run('cp -fv * /content/drive/MyDrive/GSoC_PrediCT/outputs/', shell=True, check=True)

In [ ]:
import os
import shutil
from pathlib import Path
from tqdm.notebook import tqdm

# Ensure we are in the correct directory to list processed files
os.chdir('/content/PrediCT/data/processed')

source_dir = Path('/content/PrediCT/data/processed')
dest_dir = Path('/content/drive/MyDrive/GSoC_PrediCT/data/processed')
dest_dir.mkdir(parents=True, exist_ok=True)

files_to_copy = list(source_dir.glob('*'))

print(f"Copying {len(files_to_copy)} files from {source_dir} to {dest_dir}")

for file_path in tqdm(files_to_copy, desc="Copying files"):
    # shutil.copy2 overwrites existing files by default
    shutil.copy2(file_path, dest_dir / file_path.name)
    # Uncomment the line below if you want verbose output for each file
    # print(f"Copied: {file_path.name}")

print("File copying complete.")

## Step 5 — DataLoader sanity check

Loads a small batch from the processed `.npy` volumes.

In [ ]:
mf = pd.read_csv('/content/drive/MyDrive/GSoC_PrediCT/outputs/processed_manifest.csv')
print('processed_rows=', len(mf))

print('Checking first 5 files in manifest:')
for i, row in mf.head(5).iterrows():
    path = row['processed_path']
    if os.path.exists(path):
        vol = np.load(path)
        print(f'Subject {row["subject_id"]}: shape={vol.shape}')
    else:
        print(f'File not found: {path}')
mf.head()

processed_rows= 787
Checking first 5 files in manifest:
Subject 673: shape=(138, 177, 177)
Subject 325: shape=(141, 183, 183)
Subject 548: shape=(138, 172, 172)
Subject 167: shape=(102, 191, 191)
Subject 160: shape=(144, 185, 185)


,subject_id,label,kind,processed_path,split
0,673,0,numpy,/content/PrediCT/data/processed/673.npy,train
1,325,0,numpy,/content/PrediCT/data/processed/325.npy,train
2,548,0,numpy,/content/PrediCT/data/processed/548.npy,train
3,167,0,numpy,/content/PrediCT/data/processed/167.npy,train
4,160,0,numpy,/content/PrediCT/data/processed/160.npy,train


In [ ]:
from predict.dataset import SampleRecord, VolumeDataset, build_dataloader
from predict.config import LoaderConfig
import torch.nn.functional as F

class PaddedVolumeDataset(VolumeDataset):
    def __init__(self, records, target_shape=(144, 191, 191)):
        super().__init__(records)
        self.target_shape = target_shape

    def __getitem__(self, index):
        x, y = super().__getitem__(index)
        # x shape is (C, D, H, W)
        curr_shape = x.shape[1:]
        pad = []
        for i in range(2, -1, -1): # Reverse order for F.pad (W, H, D)
            diff = self.target_shape[i] - curr_shape[i]
            pad_low = diff // 2
            pad_high = diff - pad_low
            pad.extend([pad_low, pad_high])

        x = F.pad(x, tuple(pad), value=0)
        # If slightly larger due to rounding, crop to exact target
        x = x[:, :self.target_shape[0], :self.target_shape[1], :self.target_shape[2]]
        return x, y

small = mf[mf['split'] == 'train'].head(8)
records = [
    SampleRecord(subject_id=str(r.subject_id), image=Path(r.processed_path), label=int(r.label), kind='numpy')
    for r in small.itertuples(index=False)
]

# We use the max dimensions seen in your check (approx 144, 191, 191)
ds = PaddedVolumeDataset(records, target_shape=(144, 191, 191))
dl = build_dataloader(ds, LoaderConfig(batch_size=2, shuffle=False))
x, y = next(iter(dl))
print('x.shape=', x.shape)
print('y=', y)

x.shape= torch.Size([2, 1, 144, 191, 191])
y= tensor([0, 0])


## Step 6 — Training stub (GPU if available)

This confirms a forward/backward pass works on your setup.

If `cuda_available=False` earlier, this runs on CPU.


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = (device == 'cuda')
print('device=', device)

train = mf[mf['split'] == 'train'].head(32)
records = [
    SampleRecord(subject_id=str(r.subject_id), image=Path(r.processed_path), label=int(r.label), kind='numpy')
    for r in train.itertuples(index=False)
]

# Use the same Padded dataset for training
ds = PaddedVolumeDataset(records, target_shape=(144, 191, 191))
dl = DataLoader(ds, batch_size=2, shuffle=True, num_workers=0, pin_memory=(device == 'cuda'))

model = nn.Sequential(
    nn.Conv3d(1, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool3d(2),
    nn.Conv3d(8, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool3d(1),
    nn.Flatten(),
    nn.Linear(16, 2),
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

model.train()
for step, (x, y) in enumerate(dl):
    x = x.to(device, non_blocking=True)
    y = y.to(device)
    logits = model(x)
    loss = loss_fn(logits, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    print('step', step, 'loss', float(loss))
    if step >= 5:
        break

device= cpu


/tmp/ipykernel_1900/2165543358.py:42: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('step', step, 'loss', float(loss))


step 0 loss 0.6863994598388672
step 1 loss 0.6570647954940796
step 2 loss 0.6271147131919861
step 3 loss 0.5964939594268799
step 4 loss 0.567458987236023
step 5 loss 0.5335394740104675


In [ ]:
import torch
from pathlib import Path

# Define the output path in Google Drive
model_save_path = Path('/content/drive/MyDrive/GSoC_PrediCT/outputs/model_final.pth')

# Ensure the directory exists
model_save_path.parent.mkdir(parents=True, exist_ok=True)

# Save the model weights
torch.save(model.state_dict(), model_save_path)

print(f"Model successfully saved to: {model_save_path}")

Model successfully saved to: /content/drive/MyDrive/GSoC_PrediCT/outputs/model_final.pth


## Step 7 — Inference and Deployment

Once you have trained your model and saved the weights to Google Drive, you can use it to make predictions on new data. To do this, follow these three steps:

1.  **Recreate the Model Architecture**: You must define a `nn.Module` or `nn.Sequential` block that is identical to the one used during training.
2.  **Load State Dict**: Use `torch.load()` to fetch the `.pth` file from your Drive and `load_state_dict()` to map those weights onto your model.
3.  **Evaluation Mode**: Call `.eval()` on the model. This is crucial as it disables specific layers like Dropout or Batch Normalization that behave differently during training.

Note: Ensure your input volume is preprocessed (HU windowing and resampling) and padded to the same target shape used in training (e.g., `144, 191, 191`) before passing it to the model.

In [ ]:
import torch
from torch import nn

# 1. Recreate the exact same architecture
infer_model = nn.Sequential(
    nn.Conv3d(1, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool3d(2),
    nn.Conv3d(8, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool3d(1),
    nn.Flatten(),
    nn.Linear(16, 2),
)

# 2. Load the weights
model_path = '/content/drive/MyDrive/GSoC_PrediCT/outputs/model_final.pth'
infer_model.load_state_dict(torch.load(model_path))

# 3. Set to evaluation mode
infer_model.eval()

print("Model architecture recreated and weights loaded successfully.")

# Example Inference
# with torch.no_grad():
#     output = infer_model(dummy_batch)
#     predictions = torch.argmax(output, dim=1)


Model architecture recreated and weights loaded successfully.


# Executive Summary Visuals (from saved outputs)

These cells generate charts and tables directly from the saved artifacts in `outputs/` (no full pipeline re-run required).


In [ ]:
from __future__ import annotations

from pathlib import Path
import os


def _find_outputs_dir() -> Path:
    env = os.environ.get("PREDICT_OUTPUTS_DIR", "").strip()
    if env:
        p = Path(env)
        if (p / "dataset_stats.json").exists():
            return p.resolve()

    candidates = [
        Path("outputs"),
        Path("..") / "outputs",
        Path("C:/Users/pankaj.AP-WIN11-DT/projects/PrediCT-CLI/outputs"),
        Path("C:/Users/pankaj.AP-WIN11-DT/projects/PrediCT/outputs"),
    ]
    for c in candidates:
        if (c / "dataset_stats.json").exists():
            return c.resolve()

    raise FileNotFoundError(
        "Could not locate outputs directory. Set environment variable PREDICT_OUTPUTS_DIR to the folder that contains dataset_stats.json."
    )


OUTPUTS_DIR = _find_outputs_dir()
print("OUTPUTS_DIR=", OUTPUTS_DIR)
print("files=", sorted(p.name for p in OUTPUTS_DIR.glob("*.json")))


In [ ]:
import json
import sys
import subprocess

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "matplotlib"])
    import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120})


In [ ]:
stats = json.loads((OUTPUTS_DIR / "dataset_stats.json").read_text(encoding="utf-8"))

report_path = OUTPUTS_DIR / "dicom_validation_report.csv"
mf_path = OUTPUTS_DIR / "processed_manifest.csv"

report_df = pd.read_csv(report_path) if report_path.exists() else pd.DataFrame()
mf_df = pd.read_csv(mf_path) if mf_path.exists() else pd.DataFrame()

inventoried = int(len(report_df)) if len(report_df) else int(stats.get("num_subjects", 0))
validated = int((report_df["ok"].astype(str).str.lower() == "true").sum()) if len(report_df) else inventoried
exported = int(len(mf_df)) if len(mf_df) else int(stats.get("num_subjects", 0))

print({"inventoried": inventoried, "validated": validated, "exported": exported})

fig, ax = plt.subplots(figsize=(6.5, 3.6))
labels = ["Inventoried", "Validated", "Exported"]
values = [inventoried, validated, exported]
bars = ax.bar(labels, values, color=["#6baed6", "#74c476", "#fd8d3c"])
ax.set_title("Dataset funnel")
ax.set_ylabel("Subjects")
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_ylim(0, max(values) * 1.15 if max(values) else 1)
plt.show()

split_sizes = stats.get("split_sizes") or {}
if split_sizes:
    split_order = ["train", "val", "test"]
    split_counts = [int(split_sizes.get(k, 0)) for k in split_order]
else:
    split_counts = []
    split_order = []
    if len(mf_df) and "split" in mf_df.columns:
        vc = mf_df["split"].astype(str).value_counts()
        split_order = vc.index.tolist()
        split_counts = vc.values.tolist()

if split_order:
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    bars = ax.bar([s.upper() for s in split_order], split_counts, color="#9ecae1")
    ax.set_title("Train / Validation / Test distribution")
    ax.set_ylabel("Subjects")
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_ylim(0, max(split_counts) * 1.15 if max(split_counts) else 1)
    plt.show()


In [ ]:
report_path = OUTPUTS_DIR / "dicom_validation_report.csv"
if not report_path.exists():
    raise FileNotFoundError(f"Missing validation report: {report_path}")

df = pd.read_csv(report_path)
ok_series = df["ok"].astype(str).str.lower() == "true"
total = int(len(df))
ok_n = int(ok_series.sum())
fail_n = int((~ok_series).sum())
pass_rate = float(ok_n / total) if total else 0.0

summary = pd.DataFrame(
    [
        {"Metric": "Total subjects checked", "Value": total},
        {"Metric": "Passed validation", "Value": ok_n},
        {"Metric": "Failed validation", "Value": fail_n},
        {"Metric": "Pass rate", "Value": f"{pass_rate:.2%}"},
    ]
)
display(summary)

if fail_n:
    reasons = (
        df.loc[~ok_series, "reason"]
        .fillna("unknown")
        .astype(str)
        .value_counts()
        .reset_index()
        .rename(columns={"index": "Reason", "reason": "Count"})
    )
    display(reasons.head(15))

    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    top = reasons.head(10)
    bars = ax.barh(top["Reason"].astype(str), top["Count"].astype(int), color="#fdae6b")
    ax.set_title("Top validation failure reasons")
    ax.set_xlabel("Subjects")
    ax.invert_yaxis()
    ax.bar_label(bars, padding=3, fontsize=8)
    plt.show()


In [ ]:
tf_root = OUTPUTS_DIR / "test_fixture"
raw_dir = tf_root / "raw_npy"
proc_dir = tf_root / "processed"

def _show_slice(ax, img2d, title):
    ax.imshow(img2d, cmap="gray")
    ax.set_title(title)
    ax.set_axis_off()

if raw_dir.exists() and proc_dir.exists():
    files = sorted(raw_dir.glob("*.npy"))[:5]
    n = len(files)
    fig, axes = plt.subplots(n, 2, figsize=(7.5, 2.2 * n))
    if n == 1:
        axes = np.array([axes])
    for i, f in enumerate(files):
        raw = np.load(f)
        proc = np.load(proc_dir / f.name)
        z_raw = raw.shape[0] // 2
        z_proc = proc.shape[0] // 2
        _show_slice(axes[i, 0], raw[z_raw], f"Raw (test fixture) — {f.name}")
        _show_slice(axes[i, 1], proc[z_proc], "Standardized output")
    fig.suptitle("Before vs After standardization (illustrative test fixture)")
    plt.tight_layout()
    plt.show()
else:
    mf_path = OUTPUTS_DIR / "processed_manifest.csv"
    if not mf_path.exists():
        raise FileNotFoundError(f"Missing processed manifest: {mf_path}")

    mf = pd.read_csv(mf_path)
    sample = mf.head(5)
    n = len(sample)
    fig, axes = plt.subplots(1, n, figsize=(2.4 * n, 2.8))
    if n == 1:
        axes = [axes]
    for ax, row in zip(axes, sample.itertuples(index=False), strict=False):
        p = Path(str(row.processed_path))
        vol = np.load(p)
        z = vol.shape[0] // 2
        ax.imshow(vol[z], cmap="gray")
        ax.set_title(f"{row.subject_id}")
        ax.set_axis_off()
    fig.suptitle("Standardized scan slices (exported outputs)")
    plt.tight_layout()
    plt.show()


In [ ]:
manifest = OUTPUTS_DIR / "project1" / "totalseg_heart_manifest.csv"
if manifest.exists():
    m = pd.read_csv(manifest)
    if "total_seconds" in m.columns:
        time_col = "total_seconds"
    elif "seconds" in m.columns:
        time_col = "seconds"
    else:
        time_col = None
    if time_col:
        secs = pd.to_numeric(m[time_col], errors="coerce").dropna()
        ok = m.get("ok")
        if ok is not None:
            ok_mask = ok.astype(str).str.lower().isin(["true", "1", "yes"])
            secs_ok = pd.to_numeric(m.loc[ok_mask, time_col], errors="coerce").dropna()
        else:
            secs_ok = secs

        total_s = float(secs_ok.sum()) if len(secs_ok) else 0.0
        avg_s = float(secs_ok.mean()) if len(secs_ok) else 0.0
        p50 = float(secs_ok.quantile(0.50)) if len(secs_ok) else 0.0
        p90 = float(secs_ok.quantile(0.90)) if len(secs_ok) else 0.0

        print(
            {
                "subjects_measured": int(len(secs_ok)),
                "total_runtime_hours": total_s / 3600.0,
                "avg_seconds_per_subject": avg_s,
                "p50_seconds": p50,
                "p90_seconds": p90,
            }
        )

        fig, ax = plt.subplots(figsize=(7.5, 3.6))
        ax.hist(secs_ok.values, bins=20, color="#9ecae1", edgecolor="white")
        ax.set_title("Throughput (heart mask generation) — per-subject runtime")
        ax.set_xlabel("Seconds per subject")
        ax.set_ylabel("Subjects")
        plt.show()
else:
    mf_path = OUTPUTS_DIR / "processed_manifest.csv"
    if not mf_path.exists():
        raise FileNotFoundError(f"Missing processed manifest: {mf_path}")

    mf = pd.read_csv(mf_path).head(200)
    sizes = []
    for p in mf["processed_path"].astype(str).tolist():
        fp = Path(p)
        if fp.exists():
            sizes.append(fp.stat().st_size / (1024.0 * 1024.0))

    if sizes:
        fig, ax = plt.subplots(figsize=(7.5, 3.6))
        ax.hist(sizes, bins=20, color="#bcbddc", edgecolor="white")
        ax.set_title("Exported volume size distribution (sample)")
        ax.set_xlabel("MB per subject")
        ax.set_ylabel("Subjects")
        plt.show()
